# More than a Hack 2027 - Phase 1: Ball Possession
### Full pipeline: repo analysis → data → processing → submission → evaluation

**Task.** For each of the 199,319 frames of the *tactical* video, output `H` / `A` / `D` / `X`
(home possession, away possession, dead ball, abstain). Ranked on **spellF1** (possessions matched by temporal
IoU), tie-break **macroF1**.

**What this notebook does**

| § | what |
|---|---|
| 0 | setup, locate the repo |
| 1 | analysis of the repo: frame contract, graded frames, public ground truth, scorer, baselines - **runs with no data** |
| 2 | load the gated bundle (tracking, Smart Tagging, homography) and verify alignment against the repo's own committed baseline |
| 3 | features, self-supervised models, evidence, decoder → `frame,label` CSV, validated |
| 4 | evaluation against *everything labelled that exists*, plus a stress test for the ball-visibility problem |

**Honest limits, up front**

* The only labels any participant has are the **5 public minutes (17 possessions)**. The full-match ground truth is private, so the number that
  actually ranks you cannot be computed here - §4 gives the best available substitutes and says how far to trust each.
* This notebook never trains on the public labels. They are used for *evaluation only*. (Allowed by RULES.md, but 17 possessions would overfit instantly.)
* Do not upload the data, or extracts of it, to any hosted LLM (FAQ.md / T&Cs).

## 0 · Setup

In [ ]:
import os, sys, math, time, itertools, difflib, subprocess, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.precision", 4)

# ------------------------------ USER SETTINGS ------------------------------
TEAM_NAME   = "BAU"   # submission is written to evaluation/results/<TEAM_NAME>.csv
REPO_DIR    = None        # path to the cloned repo; None = auto-detect (cwd, parents, ./more-than-a-hack-2027)
AUTO_CLONE  = True        # git clone the repo next to this notebook if it cannot be found
USE_CACHE   = True        # cache parsed tracking in data/_cache (inside the git-ignored data/ folder)
BOOTSTRAP_N = 1000        # bootstrap draws for confidence intervals
STRICT_PARITY = True      # stop if our parse of the ATD does not reproduce the repo's baseline_raw.csv
# ---------------------------------------------------------------------------

def find_repo():
    if REPO_DIR:
        return Path(REPO_DIR).expanduser().resolve()
    here = Path.cwd().resolve()
    for c in [here, *here.parents, here / "more-than-a-hack-2027"]:
        if (c / "games.py").is_file() and (c / "evaluation" / "possession_metrics.py").is_file():
            return c
    return None

REPO = find_repo()
if REPO is None and AUTO_CLONE:
    print("repo not found - cloning ...")
    subprocess.run(["git", "clone", "https://github.com/metrica-sports/more-than-a-hack-2027.git"], check=True)
    REPO = find_repo()
assert REPO is not None, "Set REPO_DIR to your clone of metrica-sports/more-than-a-hack-2027"
for sub in ("", "evaluation", "starter"):
    sys.path.insert(0, str(REPO / sub))

import games
import possession_metrics as pm
import baseline as repo_baseline          # starter/baseline.py  (team mapping + reference baseline)
import load_smart_tagging, load_homography

print("repo   :", REPO)
print("python :", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)

## 1 · Analysis of the repo (no data needed)

### 1.1 The frame contract and which frames are graded
Everything is indexed by **tactical video frame**, 0…199318. Tracking `frame_id` counts from 1 (`video_frame = frame_id − 1`); Smart Tagging is in seconds
(`round(s·25)`); homography is already 0-based. The scorer only looks at the two halves.

In [ ]:
GAME = games.GAME
N    = games.GAMES[GAME]["n_video_frames"]
FPS  = games.GAMES[GAME]["fps"]
L_M, W_M = games.GAMES[GAME]["pitch"]                    # metres (105.3 x 68.0, measured from corrected tracking)
BLOCKS = games.half_video_blocks(GAME)                   # [(1, first, last), (2, first, last)]  inclusive
RUNS   = [(a, b + 1) for _, a, b in BLOCKS]              # half-open, same convention as pm.valid_runs
GRADED = np.zeros(N, bool); HALF = np.zeros(N, np.int8)
for h, a, b in BLOCKS:
    GRADED[a:b + 1] = True; HALF[a:b + 1] = h
KICKOFF = load_smart_tagging.KICKOFF_VIDEO_FRAMES        # {'1st Half': 11185, '2nd Half': 113742}
KICK1, KICK2 = KICKOFF["1st Half"], KICKOFF["2nd Half"]
HALF2_START = BLOCKS[1][1]

print(f"frames {N}  fps {FPS}  pitch {L_M} x {W_M} m")
for h, a, b in BLOCKS:
    print(f"half {h}: video frames {a}..{b}  ({b-a+1} frames, {(b-a+1)/FPS/60:.1f} min)")
print(f"graded {GRADED.sum()} / {N}  ({100*GRADED.mean():.1f}%)")
print(f"TRAP: half 2 is graded from {HALF2_START} but kicks off at {KICK2} -> {KICK2-HALF2_START} graded frames that are all D (FAQ.md)")

### 1.2 The public ground truth - the only labels that exist
`evaluation/gt/BAR-ATH_possession_gt_public.csv`: full-length, real labels only on video frames 11185…18684 (5 min), `X` elsewhere.

In [ ]:
GT_PATH = REPO / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"
gt = pm.label_codes(pm.read_label_csv(GT_PATH))
GT_RUNS = pm.valid_runs(gt)
WIN_LO, WIN_HI = GT_RUNS[0]                                   # half-open window
print("public window:", WIN_LO, "..", WIN_HI - 1, f"({(WIN_HI-WIN_LO)/FPS/60:.1f} min)")

NAMES = "HADX"
seg = pd.DataFrame([(NAMES[k], a, b, (b - a) / FPS, (a - WIN_LO) / FPS) for a, b, k in pm.segments(gt, GT_RUNS)],
                   columns=["label", "start", "stop", "dur_s", "t_in_window_s"])
display(seg.round(1))
poss = seg[seg.label.isin(["H", "A"])]
print(f"{len(poss)} possessions | mean {poss.dur_s.mean():.1f}s | median {poss.dur_s.median():.1f}s | "
      f"shortest {poss.dur_s.min():.1f}s | longest {poss.dur_s.max():.1f}s | {(poss.dur_s<4).sum()} are shorter than 4 s")
d = seg[seg.label == "D"]
print(f"{len(d)} dead-ball stretches, {d.dur_s.min():.1f}-{d.dur_s.max():.1f}s, D share of window {d.dur_s.sum()/seg.dur_s.sum():.1%}")

### 1.3 The scorer, reproduced and instrumented
`pm.evaluate` is the same function `run_evaluation.py` calls. Helpers below turn it into tables we reuse everywhere.
Key mechanics worth internalising (all from `possession_metrics.py`):

* **Possession = maximal run of one team with ball alive.** `H H D D H H` is *two* possessions - every stoppage splits a possession, so dead-ball timing matters as much as team calls.
* **spellF1** = F1 of one-to-one matched possessions, averaged over IoU 0.3…0.7. A one-frame offset is free; 1 s late costs ~0.35.
* **`X` is forward-filled** (per half) before spellF1 - so an abstention means "repeat my last answer", cheap inside a possession, expensive across a hand-over. `macroF1` counts `X` as a miss.

In [ ]:
def score_one(gt_codes, pred_codes, bootstrap_n=0):
    return pm.evaluate(gt_codes, pred_codes, bootstrap_n=bootstrap_n)

def row_from_result(name, r):
    pc = r["per_class"]
    return dict(submission=name, spellF1=r["spell_f1_mean"], spell_lo=r["spell_bootstrap_low"], spell_hi=r["spell_bootstrap_high"],
                macroF1=r["macro_f1"], poss=r["n_pred_spells"], gt_poss=r["n_gt_spells"], dur_s=r["mean_spell_duration_s"],
                F1_H=pc["H"]["f1"], F1_A=pc["A"]["f1"], F1_D=pc["D"]["f1"], alive_dead_F1=r["alive_dead_f1"],
                team_acc=r["team_acc_both_alive"], transF1_25=r["transitions"][25]["f1"], frag=r["fragmentation"])

def score_table(preds, gt_codes, bootstrap_n=0):
    rows = [row_from_result(n, score_one(gt_codes, c, bootstrap_n)) for n, c in preds.items()]
    return pd.DataFrame(rows).sort_values(["spellF1", "macroF1"], ascending=False).reset_index(drop=True)

def possession_report(gt_codes, pred_codes, thr=0.5):
    # each true possession with the best-overlapping predicted possession of the same team
    runs = pm.valid_runs(gt_codes)
    filled = pm.fill_abstentions(pred_codes, runs)
    gs, ps = pm.possession_spells(gt_codes, runs), pm.possession_spells(filled, runs)
    ov = pm.spell_overlaps(gs, ps); hit = pm.matched_gt_flags(ov, len(gs), thr)
    best = {}
    for iou, i, j in ov:
        best.setdefault(i, (iou, j))
    rows = []
    for i, (a, b, k) in enumerate(gs):
        iou, j = best.get(i, (0.0, None))
        rows.append(dict(team=NAMES[k], start=a, dur_s=(b - a) / FPS, best_iou=iou, matched=bool(hit[i]),
                         pred_start=ps[j][0] if j is not None else None, pred_dur_s=(ps[j][1]-ps[j][0])/FPS if j is not None else None))
    return pd.DataFrame(rows)

def load_codes(path):
    return pm.label_codes(pm.read_label_csv(path))

BASELINES = {p.stem: load_codes(p) for p in sorted((REPO / "evaluation" / "results").glob("baseline_*.csv"))}
print("baselines found:", list(BASELINES))

### 1.4 Baselines on the public window - and why the board isn't a leaderboard

In [ ]:
base_tab = score_table(BASELINES, gt, BOOTSTRAP_N)
display(base_tab.round(3))
print("Same algorithm four times, only the smoothing window changes. macroF1 barely moves; spellF1 moves ~11x.")
print("Published FULL-MATCH spellF1 (EVALUATION.md): raw 0.0486 | smooth25 0.1996 | smooth50 0.2433 | smooth125 0.2242")
print("-> on the public window smooth125 'wins' (0.42) but on the full match it is worse than smooth50. 17 possessions cannot rank anything.")

### 1.5 What the shipped baseline files reveal about the *whole match* (still no data needed)
`baseline_raw.csv` labels `D` exactly where the ball is not tracked, so it doubles as a **ball-visibility mask for all 199,319 frames**.

In [ ]:
raw = BASELINES["baseline_raw"]
ball_missing_mask = (raw == pm.D)
def run_bounds(mask):
    d = np.diff(np.concatenate(([0], np.asarray(mask).astype(np.int8), [0])))
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1)

vis_graded = 1 - ball_missing_mask[GRADED].mean()
vis_window = 1 - ball_missing_mask[WIN_LO:WIN_HI].mean()
gaps = np.concatenate([np.diff(np.stack(run_bounds(ball_missing_mask[a:b])), axis=0).ravel() for a, b in RUNS])
print(f"ball visible: graded frames {vis_graded:.1%}  |  public window {vis_window:.1%}   <- the window is the EASIEST part of the match")
print(f"ball-missing gaps in graded frames: n={len(gaps)}  median {np.median(gaps):.0f} fr ({np.median(gaps)/FPS:.1f}s)  "
      f"mean {gaps.mean()/FPS:.1f}s  p90 {np.percentile(gaps,90)/FPS:.0f}s  max {gaps.max()/FPS:.0f}s")
for t in (25, 125, 250, 500):
    print(f"  missing frames that live in gaps <= {t/FPS:4.0f}s : {gaps[gaps<=t].sum()/gaps.sum():6.1%}")
print(f"  missing frames in gaps > 20s: {gaps[gaps>500].sum()/gaps.sum():.1%}  ({(gaps>500).sum()} gaps)")
print("=> interpolating the ball across 'brief occlusions' is nearly useless: gaps are long. Possession in them must come from players + Smart Tagging.")

blk = []
for h, (a, b) in enumerate(RUNS, 1):
    blk += [(h, (s - a) / FPS / 60, 100 * (1 - ball_missing_mask[s:min(s + 7500, b)].mean())) for s in range(a, b, 7500)]
vis_blocks = pd.DataFrame(blk, columns=["half", "minute", "ball_visible_%"])
print("\nball visibility per 5-minute block:")
print(vis_blocks.pivot_table(index="half", columns=vis_blocks.groupby("half").cumcount(), values="ball_visible_%").round(0).astype(int).to_string())

print("\nfull-match possession counts of the shipped baselines:")
for n, c in BASELINES.items():
    sp = pm.possession_spells(c, RUNS); L = np.array([b - a for a, b, _ in sp]) / FPS
    print(f"  {n:20s} {len(sp):5d} possessions, mean {L.mean():5.1f}s, median {np.median(L):4.1f}s")

in_win = (gt[WIN_LO:WIN_HI] != pm.D) & (raw[WIN_LO:WIN_HI] != pm.D)
acc = (gt[WIN_LO:WIN_HI][in_win] == raw[WIN_LO:WIN_HI][in_win]).mean()
print(f"\nnearest-player-to-ball team == GT team on alive & ball-visible window frames: {acc:.1%}  (so even a clean ball leaves ~22% wrong)")
gtD = gt[WIN_LO:WIN_HI] == pm.D
print(f"GT dead-ball frames with the ball visible: {(raw[WIN_LO:WIN_HI][gtD] != pm.D).mean():.1%}  (so 'ball missing => D' is wrong most of the time)")

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
    for h, g in vis_blocks.groupby("half"):
        ax[0].plot(g.minute + (0 if h == 1 else 0), g["ball_visible_%"], marker="o", label=f"half {h}")
    ax[0].axhline(100 * vis_graded, ls="--", c="grey"); ax[0].set_ylim(0, 100)
    ax[0].set_title("ball visible % per 5-min block (public window = first point)"); ax[0].set_xlabel("minute in half"); ax[0].legend()
    ax[1].hist(gaps / FPS, bins=40); ax[1].set_title("ball-missing gap lengths (s)"); ax[1].set_xlabel("seconds")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("plot skipped:", e)

## 2 · Load the gated data bundle

Unpack the download into `data/` (or set `MTAH_DATA_DIR`). If the files are missing this section stops with a readable message - everything above still ran.

**Parsing choices.** We read the raw EPTS text ourselves with numpy (no kloppy needed): lines look like `frame_id:slot;slot;…:ball`, each group `x,y,speed`, coordinates normalised 0–1.
We then **prove the parser, team mapping and frame alignment are right** by reproducing the repo's committed `baseline_raw.csv` from our arrays - that file was produced from the same ATD by `starter/baseline.py`.

In [ ]:
PATHS = games.asset_paths()
NEED = ["atd_meta", "atd_raw", "homography", "smart_tagging"]
missing = [PATHS[k] for k in NEED if not os.path.isfile(PATHS[k])]
DATA_OK = not missing
if not DATA_OK:
    print("DATA NOT FOUND:\n  " + "\n  ".join(missing))
    print("\nEnrol at https://landing.mobileworldcapital.com/more-than-a-hack-2027, unpack into", games.DATA_DIR,
          "or set MTAH_DATA_DIR, then re-run from here.")

def require_data():
    if not DATA_OK:
        raise RuntimeError("Match data not found - see the message above. Sections 0-1 do not need it.")
require_data()
for k in NEED:
    print(f"{k:14s} {os.path.getsize(PATHS[k])/1e6:8.1f} MB   {PATHS[k]}")

In [ ]:
require_data()
import xml.etree.ElementTree as ET

def describe_metadata(meta_path, max_lines=40):
    # What channels does the ATD declare? (Does the ball carry anything beyond x,y,speed?)
    root = ET.parse(meta_path).getroot()
    counts = {}
    for el in root.iter():
        t = el.tag.split("}")[-1]; counts[t] = counts.get(t, 0) + 1
    print("element counts:", {k: v for k, v in sorted(counts.items(), key=lambda kv: -kv[1])[:14]})
    shown = 0
    for el in root.iter():
        t = el.tag.split("}")[-1]
        if ("Channel" in t or "Spec" in t or "Unit" in t) and shown < max_lines:
            print("  ", t, dict(el.attrib), (el.text or "").strip()[:60]); shown += 1

describe_metadata(PATHS["atd_meta"])
with open(PATHS["atd_raw"]) as fh:
    for _ in range(3): last = fh.readline()
print("\nsample raw line (truncated):", last[:230].rstrip(), "...", last[-90:].rstrip())

In [ ]:
require_data()
_TRANS = str.maketrans({":": " ", ";": " ", ",": " "})

def _probe_layout(path, stride=997):
    n_slots = n_pf = n_bf = 0
    with open(path) as fh:
        for i, line in enumerate(fh):
            if i % stride: continue
            parts = line.rstrip("\n").split(":")
            if len(parts) != 3: continue
            groups = parts[1].split(";")
            n_slots = max(n_slots, len(groups))
            n_pf = max([n_pf] + [len(g.split(",")) for g in groups])
            n_bf = max(n_bf, len(parts[2].split(",")))
    return n_slots, n_pf, n_bf

def _parse_slow(lines, n_slots, n_pf, n_bf, width):
    out = np.full((len(lines), width), np.nan)
    for r, line in enumerate(lines):
        parts = line.rstrip("\n").split(":")
        if len(parts) != 3: continue
        out[r, 0] = float(parts[0])
        for s, g in enumerate(parts[1].split(";")[:n_slots]):
            if g and g[0] != "N":
                v = g.split(",")
                if len(v) >= n_pf: out[r, 1 + s*n_pf: 1 + (s+1)*n_pf] = [float(x) for x in v[:n_pf]]
        g = parts[2]
        if g and g[0] != "N":
            v = g.split(",")
            if len(v) >= n_bf: out[r, 1 + n_slots*n_pf:] = [float(x) for x in v[:n_bf]]
    return out

def parse_atd(path, n_frames, chunk=20000):
    n_slots, n_pf, n_bf = _probe_layout(path)
    width = 1 + n_slots * n_pf + n_bf
    P = np.full((n_frames, n_slots, n_pf), np.nan, np.float32)
    B = np.full((n_frames, n_bf), np.nan, np.float32)
    seen = np.zeros(n_frames, bool); slow = 0
    with open(path) as fh:
        while True:
            lines = list(itertools.islice(fh, chunk))
            if not lines: break
            arr = None
            try:
                a = np.fromstring("".join(lines).translate(_TRANS), sep=" ")
                if a.size == len(lines) * width: arr = a.reshape(len(lines), width)
            except Exception:
                arr = None
            if arr is None:
                arr = _parse_slow(lines, n_slots, n_pf, n_bf, width); slow += 1
            ids = np.where(np.isnan(arr[:, 0]), -1, arr[:, 0] - 1).astype(np.int64)
            ok = (ids >= 0) & (ids < n_frames)
            ids, arr = ids[ok], arr[ok]
            P[ids] = arr[:, 1: 1 + n_slots*n_pf].reshape(-1, n_slots, n_pf)
            B[ids] = arr[:, 1 + n_slots*n_pf:]
            seen[ids] = True
    return P, B, seen, dict(n_slots=n_slots, n_player_fields=n_pf, n_ball_fields=n_bf, slow_chunks=slow)

CACHE = Path(games.DATA_DIR) / "_cache"; CACHE.mkdir(parents=True, exist_ok=True)
cache_file = CACHE / "atd_tactical.npz"
t0 = time.time()
if USE_CACHE and cache_file.is_file() and cache_file.stat().st_mtime > os.path.getmtime(PATHS["atd_raw"]):
    z = np.load(cache_file, allow_pickle=True); P_RAW, B_RAW, SEEN, LAYOUT = z["P"], z["B"], z["seen"], z["layout"].item()
    print("loaded from cache")
else:
    P_RAW, B_RAW, SEEN, LAYOUT = parse_atd(PATHS["atd_raw"], N)
    if USE_CACHE: np.savez(cache_file, P=P_RAW, B=B_RAW, seen=SEEN, layout=np.array(LAYOUT, dtype=object))
print(f"parsed in {time.time()-t0:.0f}s  layout {LAYOUT}")
print(f"records present {SEEN.sum()} / {N}  -> one record per video frame, ids 1..N: {bool(SEEN.all())}")
if not SEEN.all(): print("WARNING: gaps in the tracking record ids - do not invent a correction, check DATA.md alignment")
if LAYOUT["n_ball_fields"] > 3: print("NOTE: the ball has", LAYOUT["n_ball_fields"], "fields - inspect B_RAW[:, 3:], it may carry status/ownership")

In [ ]:
require_data()
# ---- team membership of each track slot (first team in the metadata = home, the FIFA/EPTS convention) ----
TEAM_OF_SLOT = repo_baseline.team_of_each_track(PATHS["atd_meta"])         # 0 home / 1 away, one entry per slot
HOME_NAME, AWAY_NAME = repo_baseline.team_names(PATHS["atd_meta"])
assert len(TEAM_OF_SLOT) == P_RAW.shape[1], (len(TEAM_OF_SLOT), P_RAW.shape)
print(f"H = {HOME_NAME}  ({(TEAM_OF_SLOT==0).sum()} slots) | A = {AWAY_NAME}  ({(TEAM_OF_SLOT==1).sum()} slots)")

PX = P_RAW[:, :, 0].astype(np.float64) * L_M          # metres
PY = P_RAW[:, :, 1].astype(np.float64) * W_M
PS = P_RAW[:, :, 2].astype(np.float64) if P_RAW.shape[2] > 2 else np.full(PX.shape, np.nan)   # m/s as delivered
BX = B_RAW[:, 0].astype(np.float64) * L_M
BY = B_RAW[:, 1].astype(np.float64) * W_M
VIS = ~np.isnan(BX)

n_obj = (~np.isnan(PX)).sum(1)
print(f"ball visible: all frames {VIS.mean():.1%} | graded {VIS[GRADED].mean():.1%} | public window {VIS[WIN_LO:WIN_HI].mean():.1%}")
print(f"tracked objects per graded frame: mean {n_obj[GRADED].mean():.1f}  (>22 on {(n_obj[GRADED]>22).mean():.1%} of frames = duplicated tracks)")
print(f"tracked objects at halftime (frame 100000): {n_obj[100000]}")

# ---- PARITY CHECK: reproduce the repo's committed baseline_raw.csv from our own arrays ----
def nearest_team_codes(PX, PY, BX, BY, team_of_slot, rows=None):
    n = len(BX); lo, hi = (0, n) if rows is None else rows
    out = np.full(n, 2, np.int8)
    dx = PX[lo:hi] - BX[lo:hi, None]; dy = PY[lo:hi] - BY[lo:hi, None]
    d2 = dx*dx + dy*dy
    d2 = np.where(np.isnan(d2), np.inf, d2)
    idx = d2.argmin(1); best = d2[np.arange(hi - lo), idx]
    out[lo:hi] = np.where(np.isfinite(best), team_of_slot[idx], 2)
    return out

mine = nearest_team_codes(PX, PY, BX, BY, TEAM_OF_SLOT)
agree = (mine == BASELINES["baseline_raw"])
print(f"\nPARITY vs committed baseline_raw.csv: {agree.mean():.4%} of all frames identical ({(~agree).sum()} differ)")
if agree.mean() > 0.999:
    print("  OK - parser, team mapping and frame alignment reproduce the repo's own result.")
else:
    msg = "MISMATCH - our arrays do not reproduce baseline_raw.csv. Check team order / frame offset / parser before trusting anything below."
    if STRICT_PARITY: raise RuntimeError(msg + " (set STRICT_PARITY=False to continue anyway)")
    print("  " + msg)

In [ ]:
require_data()
# ---- Smart Tagging ----
TAGS = load_smart_tagging.load()
tag_report = load_smart_tagging.check_time_base(TAGS)
print(f"{len(TAGS)} tag instances | time base agrees with the real kickoffs: {tag_report['agrees']}")
tag_raw = pd.DataFrame([{"code": t["code"], "s": t["start_frame"], "e": t["end_frame"], **t["tags"]} for t in TAGS])
print(tag_raw.code.value_counts().to_string())
print("Team tag values:", tag_raw["Team"].value_counts().to_dict())

# ---- Homography ----
HMAT, HERR, HVALID = load_homography.load()
print(f"\nhomography: rows {len(HVALID)} (== N: {len(HVALID)==N}), valid {HVALID.mean():.1%}, "
      f"median reprojection error during play {np.median(HERR[GRADED]):.3f}")

## 3 · Processing: from tracking + tags to a possession label per frame

### Design (and why)
The two facts from §1.5 drive everything: **the ball is missing in long gaps (41% of graded frames, 67% of that in gaps > 20 s)**, and **every stoppage splits a possession**.
So the method is a small probabilistic decoder over *evidence*, not a nearest-player vote:

1. **Dead ball vs alive** - decoded first, because a dead stretch ends a possession.
   Evidence: ball outside the pitch; ball stationary ≥ 3 s; a **player-only model** (learned *without any ground truth*, see below) for frames with no ball;
   Smart-Tagging set-piece windows (restarts ⇒ dead until the ball moves). Rules from the docs: the first 2,934 frames of half 2 are `D`; half 1 is graded from kickoff.
2. **Team inside each alive stretch** - a 2-state Viterbi over log-odds evidence with a **switch penalty** (in nat·seconds). A flip needs *accumulated* evidence, which is the
   "controlled possession" idea from CHALLENGE.md: a loose ball or a deflection leaves possession where it was.
   Evidence: `κ·(d_away − d_home)` from the ball (gated to 0 when the ball is far from everyone, i.e. in flight); Smart-Tagging phases with the correct *possession semantics*
   (Build Up/Progression/… ⇒ that team has it; **Defending…/Defensive Transition ⇒ the *opponent* has it**); and a **team-shape model** for ball-less frames.
3. **Self-supervised player-only models.** On frames where the ball *is* visible we derive confident pseudo-labels from ball kinematics/proximity, train two small logistic models that see
   **only players** (team centroids/spreads/speeds/advance in attacking direction), and apply them where the ball is missing. No ground truth, public or private, is used.

Every threshold lives in `CFG` below. They are set from football/physics priors, **not tuned on the public window** (see §4 for why that would be a mistake).

In [ ]:
CFG = dict(
    # ball -> team evidence
    kappa=0.8, clip=4.0, r_gate=7.0,
    # smart tagging
    w_tag=0.8, tag_cap=2.0,
    # player-only team model
    w_shape=0.6, shape_clip=2.0,
    # decoder
    lam_team=2.5,                  # nat*s needed to flip possession (higher = fewer, longer possessions)
    lam_dead=3.0,                  # nat*s needed to enter/leave dead ball
    min_dead_s=2.0, min_alive_s=1.5,
    # dead-ball evidence
    v_static=1.0, static_min_s=3.0, oob_margin=1.0, oob_min_s=0.5, v_alive=3.0,
    kin_dead=4.0, kin_alive=3.0, w_pm=0.5, pm_clip=3.0, w_sp=2.0, v_restart=3.0, sp_fallback=0.5,
    # pseudo-labels for self-supervision
    ctrl_dist=1.5, ctrl_gap=2.5,
)

# ---------------- small numeric helpers ----------------
def keep_runs(mask, min_len):
    out = np.zeros(len(mask), bool); s, e = run_bounds(mask)
    for a, b in zip(s, e):
        if b - a >= min_len: out[a:b] = True
    return out

def enforce_min_runs(dead, min_dead, min_alive):
    # drop dead runs shorter than min_dead, then fill alive gaps shorter than min_alive between dead runs
    dead = dead.copy(); s, e = run_bounds(dead)
    for a, b in zip(s, e):
        if b - a < min_dead: dead[a:b] = False
    s, e = run_bounds(~dead)
    for a, b in zip(s, e):
        if b - a < min_alive and a > 0 and b < len(dead): dead[a:b] = True
    return dead

def moving_mean(x, half):
    m = ~np.isnan(x); v = np.where(m, x, 0.0)
    cs = np.concatenate(([0.0], np.cumsum(v))); cm = np.concatenate(([0.0], np.cumsum(m)))
    idx = np.arange(len(x)); lo = np.maximum(0, idx - half); hi = np.minimum(len(x), idx + half + 1)
    num, den = cs[hi] - cs[lo], cm[hi] - cm[lo]
    return np.where(den > 0, num / np.maximum(den, 1), np.nan)

def shift_diff(x, k):
    out = np.full(len(x), np.nan); out[k:-k] = x[2*k:] - x[:-2*k]; return out

def viterbi2(score0, score1, penalty):
    # 2-state Viterbi; score* = per-frame score of each state (higher is better); switching costs `penalty`
    n = len(score0)
    if n == 0: return np.zeros(0, np.int8)
    back = np.zeros((n, 2), np.int8); s0, s1 = float(score0[0]), float(score1[0])
    for t in range(1, n):
        b0 = s1 - penalty
        if b0 > s0: back[t, 0] = 1; n0 = b0
        else: n0 = s0
        b1 = s0 - penalty
        if b1 > s1: back[t, 1] = 0; n1 = b1
        else: back[t, 1] = 1; n1 = s1
        s0 = n0 + score0[t]; s1 = n1 + score1[t]
    st = np.empty(n, np.int8); st[-1] = 0 if s0 >= s1 else 1
    for t in range(n - 1, 0, -1): st[t - 1] = back[t, st[t]]
    return st

def fit_logreg(X, y, l2=3.0, iters=40):
    n, d = X.shape; Xb = np.hstack([np.ones((n, 1)), X]); w = np.zeros(d + 1)
    R = np.eye(d + 1) * l2; R[0, 0] = 0
    for _ in range(iters):
        p = 1 / (1 + np.exp(-np.clip(Xb @ w, -30, 30)))
        g = Xb.T @ (p - y) + R @ w
        Hs = (Xb * (p * (1 - p))[:, None]).T @ Xb + R + 1e-6 * np.eye(d + 1)
        step = np.linalg.solve(Hs, g); w -= step
        if np.max(np.abs(step)) < 1e-6: break
    return w

def predict_logreg(w, X):
    return 1 / (1 + np.exp(-np.clip(w[0] + X @ w[1:], -30, 30)))

def auc(score, y):
    r = pd.Series(score).rank().values; n1 = y.sum(); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 and n0 else float("nan")

def logit(p): p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))

### 3.1 Attack direction
The ATD is in absolute pitch coordinates, but "advancing" only means something relative to the direction a team is attacking. We read it from the team centroids at each kickoff
(no assumption about which way Barcelona kicks), and cross-check that the second half is the mirror of the first.

In [ ]:
require_data()
def home_centroid_x(frames):
    sel = TEAM_OF_SLOT == 0
    return float(np.nanmean(PX[frames][:, sel] / L_M))

c1, c2 = home_centroid_x(slice(KICK1, KICK1 + 50)), home_centroid_x(slice(KICK2, KICK2 + 50))
D1, D2 = (1 if c1 < 0.5 else -1), (1 if c2 < 0.5 else -1)
print(f"home centroid x at kickoffs: half1 {c1:.2f} -> attacks {'+x' if D1>0 else '-x'} | half2 {c2:.2f} -> attacks {'+x' if D2>0 else '-x'}")
print("sides swap at half time:", D2 == -D1)
halfmean = [home_centroid_x(slice(a, b + 1)) for _, a, b in BLOCKS]
print("sanity (home centroid averaged over each half; attacks +x => expect < 0.5 in own-half terms):", np.round(halfmean, 3))
assert D2 == -D1, "directions do not mirror - inspect kickoff frames"
DIRV = np.where(np.arange(N) < (BLOCKS[0][2] + BLOCKS[1][1]) // 2, D1, D2).astype(np.int8)

### 3.2 Ball-dependent features and self-supervised pseudo-labels
Everything that depends on the ball lives in one function so the §4 stress test can re-run it with a degraded ball.

In [ ]:
def ball_features(BXm, BYm, rows=None):
    n = len(BXm); lo, hi = (0, n) if rows is None else rows
    a, b = max(0, lo - 100), min(n, hi + 100)
    out = {k: np.full(n, np.nan) for k in ("dH", "dA", "bspeed")}
    bx, by, px, py = BXm[a:b], BYm[a:b], PX[a:b], PY[a:b]
    for key, tm in (("dH", 0), ("dA", 1)):
        sel = TEAM_OF_SLOT == tm
        d = np.sqrt((px[:, sel] - bx[:, None])**2 + (py[:, sel] - by[:, None])**2)
        d = np.where(np.isnan(d), np.inf, d).min(1); d[~np.isfinite(d)] = np.nan
        out[key][a:b] = d
    h = 12                                              # speed from +-0.5 s displacement: robust to tracker jitter
    sp = np.full(b - a, np.nan)
    if b - a > 2 * h:
        sp[h:-h] = np.hypot(bx[2*h:] - bx[:-2*h], by[2*h:] - by[:-2*h]) / (2 * h / FPS)
    out["bspeed"][a:b] = sp
    out["vis"] = ~np.isnan(BXm); out["BX"], out["BY"] = BXm, BYm
    out["dmin"] = np.fmin(out["dH"], out["dA"])
    return out

def pseudo_labels(bf, cfg):
    m = cfg["oob_margin"]
    oob = bf["vis"] & ((bf["BX"] < -m) | (bf["BX"] > L_M + m) | (bf["BY"] < -m) | (bf["BY"] > W_M + m))
    oob_run = keep_runs(oob, int(cfg["oob_min_s"] * FPS))
    still = np.isfinite(bf["bspeed"]) & (bf["bspeed"] < cfg["v_static"])
    still_run = keep_runs(still, int(cfg["static_min_s"] * FPS))
    dead_ps = oob_run | still_run
    alive_ps = np.isfinite(bf["bspeed"]) & (bf["bspeed"] > cfg["v_alive"]) & ~oob & ~dead_ps
    dH, dA = np.nan_to_num(bf["dH"], nan=99.0), np.nan_to_num(bf["dA"], nan=99.0)
    ctrl = bf["vis"] & ~dead_ps & (np.minimum(dH, dA) < cfg["ctrl_dist"]) & (np.abs(dA - dH) > cfg["ctrl_gap"])
    return dict(oob=oob, dead_ps=dead_ps, alive_ps=alive_ps, ctrl=ctrl, ctrl_home=(dH < dA))

BF_FULL = ball_features(BX, BY)
PS_FULL = pseudo_labels(BF_FULL, CFG)
g = GRADED
print(f"pseudo-labels on graded frames: dead {PS_FULL['dead_ps'][g].sum()} | alive(fast ball) {PS_FULL['alive_ps'][g].sum()} | "
      f"controlled-by-a-team {PS_FULL['ctrl'][g].sum()} ({PS_FULL['ctrl'][g & PS_FULL['ctrl']].size/ max(1,g.sum()):.1%} of graded)")

# QC against the public GT (window only; evaluation, not training)
w = slice(WIN_LO, WIN_HI); G = gt[w]
dp, ap, cl = PS_FULL["dead_ps"][w], PS_FULL["alive_ps"][w], PS_FULL["ctrl"][w]
print("\npseudo-label quality on the public window (a correctness check, not a tuning target):")
print(f"  dead pseudo-label : precision {(G[dp]==pm.D).mean():.2f}  recall of GT-D {dp[G==pm.D].mean():.2f}   (n={dp.sum()})")
print(f"  fast-ball 'alive' : precision {(G[ap]!=pm.D).mean():.2f}                                 (n={ap.sum()})")
ch = PS_FULL["ctrl_home"][w]
ok = cl & (G != pm.D)
print(f"  controlled-team   : team agrees with GT on {((ch[ok]==(G[ok]==pm.H))).mean():.2f} of {ok.sum()} frames")

### 3.3 Player-only features and the two self-supervised models
Features use players only - never the ball - so they remain defined in the 41% of frames where the ball is missing.
Quality is reported by **time-blocked cross-validation against the pseudo-labels** (AUC). This tells us whether the player-only signal exists at all; it is *not* an accuracy estimate for the final labels.

In [ ]:
def build_player_features():
    Xn, Yn = PX / L_M, PY / W_M
    Xo = np.where((DIRV == -1)[:, None], 1 - Xn, Xn)           # oriented so HOME always attacks +x
    V = np.clip(PS, 0, 12); f = {}
    for nm, tm in (("H", 0), ("A", 1)):
        s = TEAM_OF_SLOT == tm
        f["cx_" + nm] = np.nanmean(Xo[:, s], 1); f["cy_" + nm] = np.nanmean(Yn[:, s], 1)
        f["sx_" + nm] = np.nanstd(Xo[:, s], 1);  f["sy_" + nm] = np.nanstd(Yn[:, s], 1)
        f["v_" + nm] = np.nanmean(V[:, s], 1)
    f["ax"], f["ay"] = np.nanmean(Xo, 1), np.nanmean(Yn, 1)
    f["ssx"], f["ssy"] = np.nanstd(Xo, 1), np.nanstd(Yn, 1)
    f["v_all"], f["v_med"] = np.nanmean(V, 1), np.nanmedian(V, 1)
    f["slow"] = np.nanmean(np.where(np.isnan(V), np.nan, (V < 1.0).astype(float)), 1)
    f["ntrk"] = (~np.isnan(PX)).sum(1).astype(float)
    return f

PF = build_player_features()

def dead_matrix():
    cols = []
    for k in ("v_all", "slow", "v_med"):
        for hw in (12, 50, 125): cols.append(moving_mean(PF[k], hw))
    for k in ("ssx", "ssy", "ntrk"): cols.append(moving_mean(PF[k], 25))
    cols += [np.abs(moving_mean(PF["ax"], 25) - 0.5), np.abs(moving_mean(PF["ay"], 25) - 0.5)]
    return np.column_stack(cols)

def team_matrix():
    S = {k: moving_mean(v, 12) for k, v in PF.items()}
    cols = [S["cx_H"], S["cx_A"], S["cx_H"] - S["cx_A"], S["cy_H"], S["cy_A"], S["sx_H"], S["sx_A"], S["sy_H"], S["sy_A"],
            S["v_H"], S["v_A"], S["v_H"] - S["v_A"], S["ax"], S["ay"],
            shift_diff(S["cx_H"], 50), shift_diff(S["cx_A"], 50), shift_diff(S["ax"], 50)]
    return np.column_stack(cols)

def standardise(M):
    mu, sd = np.nanmean(M[GRADED], 0), np.nanstd(M[GRADED], 0) + 1e-9
    return np.nan_to_num((M - mu) / sd)

XD, XT = standardise(dead_matrix()), standardise(team_matrix())

# time blocks over graded frames for honest cross-validation
order = np.flatnonzero(GRADED); BLK = np.full(N, -1); BLK[order] = np.arange(len(order)) * 6 // len(order)
rng = np.random.RandomState(0)

def fit_selfsup(X, y_pos, y_neg, name, max_rows=80000):
    lab = (y_pos | y_neg) & GRADED
    yy = y_pos.astype(float)
    n1, n0 = int((y_pos & lab).sum()), int((y_neg & lab).sum())
    print(f"[{name}] pseudo-labelled frames: positive {n1}, negative {n0}")
    if min(n1, n0) < 300:
        print(f"[{name}] too few labels -> model disabled (constant 0.5)"); return np.full(N, 0.5)
    idx = np.flatnonzero(lab)
    if len(idx) > max_rows: idx = rng.choice(idx, max_rows, replace=False)
    aucs = []
    for k in range(6):
        tr, te = idx[BLK[idx] != k], idx[BLK[idx] == k]
        if min(yy[te].sum(), len(te) - yy[te].sum()) < 20 or min(yy[tr].sum(), len(tr) - yy[tr].sum()) < 20: continue
        w_ = fit_logreg(X[tr], yy[tr]); aucs.append(auc(predict_logreg(w_, X[te]), yy[te]))
    print(f"[{name}] block-CV AUC vs pseudo-labels: {np.round(aucs, 3)}  mean {np.mean(aucs) if aucs else float('nan'):.3f}")
    return predict_logreg(fit_logreg(X[idx], yy[idx]), X)

P_DEAD_PM = fit_selfsup(XD, PS_FULL["dead_ps"], PS_FULL["alive_ps"], "dead-ball model (players only)")
P_HOME_SHAPE = fit_selfsup(XT, PS_FULL["ctrl"] & PS_FULL["ctrl_home"], PS_FULL["ctrl"] & ~PS_FULL["ctrl_home"], "team-shape model (players only)")

# applied where the ball is missing: how does it do on the public window? (evaluation only)
miss_w = ~VIS[w]; alive_g = (G != pm.D)
m1 = miss_w & alive_g
if m1.sum() > 50:
    print(f"\nteam-shape model on public-window frames with NO ball and GT alive (n={m1.sum()}): "
          f"agrees with GT on {((P_HOME_SHAPE[w][m1] > 0.5) == (G[m1] == pm.H)).mean():.1%}  (chance ~ 50%)")
m2 = miss_w
if m2.sum() > 50 and (G[m2] == pm.D).any():
    print(f"dead-ball model on public-window frames with NO ball (n={m2.sum()}): AUC vs GT dead {auc(P_DEAD_PM[w][m2], (G[m2]==pm.D).astype(int)):.3f}")

### 3.4 Smart Tagging → evidence
Mapping rules (from OVERVIEW.md; *Team = the team performing the event*):

| code | implies |
|---|---|
| Build Up, Progression, Creating Chances, Attacking Transition, Long Ball, Ball in Final Third, Ball in the Box, Players in the Box, Goals | **that team** has the ball |
| Defending in Attacking/Middle/Defensive Third, Defensive Transition | **the opponent** has the ball |
| Set Pieces | a **restart**: ball dead until it moves (Throw-ins have `Team = N/A`) |

Tags are *phases, not possessions* and are automatic (false positives/negatives) - they enter as modest, capped evidence, never as labels.

In [ ]:
def team_sign(txt):
    if txt is None: return 0
    t = txt.strip().lower()
    if t in ("", "n/a", "na"): return 0
    if "barc" in t: return 1                                   # DATA.md: home is Barcelona
    if "ath" in t or "atl" in t: return -1                     # the file spells it 'Atletic Club'
    rh = difflib.SequenceMatcher(None, t, HOME_NAME.lower()).ratio(); ra = difflib.SequenceMatcher(None, t, AWAY_NAME.lower()).ratio()
    return 0 if max(rh, ra) < 0.6 or abs(rh - ra) < 0.05 else (1 if rh > ra else -1)

TAGDF = pd.DataFrame([dict(code=(t["code"] or "").strip().lower(), s=max(0, t["start_frame"]), e=min(N, t["end_frame"]),
                           team=team_sign(t["tags"].get("Team")), typ=(t["tags"].get("Type") or "").strip().lower()) for t in TAGS])
print("team mapping of tags:", TAGDF.team.value_counts().to_dict(), "(+1 home, -1 away, 0 = N/A or unmapped)")

CODE_W = {"build up": 1.0, "progression": 1.0, "creating chances": 1.0, "attacking transition": 0.8, "long ball": 0.6,
          "ball in final third": 1.0, "ball in the box": 1.0, "players in the box": 1.0, "goals": 1.0,
          "defending in attacking third": -0.8, "defending in middle third": -0.8, "defending in defensive third": -0.8,
          "defensive transition": -0.5}

def tag_team_raw(tagdf):
    L = np.zeros(N)
    for r in tagdf.itertuples():
        w_ = CODE_W.get(r.code)
        if w_ is not None and r.team != 0: L[r.s:r.e] += w_ * r.team
    return L

def set_piece_prior(tagdf, bspeed, cfg):
    pre = np.zeros(N, bool); info = []
    for r in tagdf[tagdf.code == "set pieces"].itertuples():
        if "kick off (start)" in r.typ: continue                 # half is graded from the kickoff itself
        s, e = r.s, r.e
        if e <= s: continue
        mv = np.flatnonzero(np.nan_to_num(bspeed[s:e], nan=0.0) > cfg["v_restart"])
        restart = s + int(mv[0]) if len(mv) else s + int(cfg["sp_fallback"] * (e - s))
        pre[s:restart] = True; info.append((r.typ, s, e, restart, len(mv) > 0))
    return pre, info

L_TAG_RAW = tag_team_raw(TAGDF)
L_TAG = np.clip(CFG["w_tag"] * L_TAG_RAW, -CFG["tag_cap"], CFG["tag_cap"])
SP_PRE_FULL, SP_INFO = set_piece_prior(TAGDF, BF_FULL["bspeed"], CFG)
print(f"set pieces: {len(SP_INFO)} restarts; ball-speed restart found for {sum(i[4] for i in SP_INFO)} of them, fallback midpoint for the rest")

# ---- how do the tags line up with the PUBLIC ground truth? (read this - it validates the semantics above) ----
sgn = np.sign(L_TAG_RAW[w]); okm = (sgn != 0) & (G != pm.D)
if okm.sum():
    print(f"\ntag-implied team vs GT team on public-window frames inside team-bearing tags: {((sgn[okm] > 0) == (G[okm] == pm.H)).mean():.1%} (n={okm.sum()})")
rows = []
for r in TAGDF[(TAGDF.code == "set pieces") & (TAGDF.e > WIN_LO) & (TAGDF.s < WIN_HI)].itertuples():
    seg_ = gt[max(r.s, WIN_LO):min(r.e, WIN_HI)]; c = np.bincount(seg_, minlength=4)
    rows.append(dict(type=r.typ, start=r.s, end=r.e, GT_H=c[0], GT_A=c[1], GT_D=c[2], GT_X=c[3]))
print("\nset-piece tags inside the public window and what the ground truth says over them:")
display(pd.DataFrame(rows) if rows else "none in window")

### 3.5 Decoder
`decode(F, …)` = dead/alive Viterbi → minimum-run clean-up → forced rules from the docs → per-stretch team Viterbi.
Frames outside the two halves are `X` (never scored). Inside the halves every frame gets a real label: `X` there would cost macroF1 and buys nothing on spellF1 beyond what the Viterbi already gives.

In [ ]:
STATIC = dict(P_DEAD_PM=P_DEAD_PM, P_HOME_SHAPE=P_HOME_SHAPE, L_TAG=L_TAG)

def make_F(BXm=None, BYm=None, rows=None, cfg=CFG):
    BXm = BX if BXm is None else BXm; BYm = BY if BYm is None else BYm
    bf = ball_features(BXm, BYm, rows); ps = pseudo_labels(bf, cfg)
    sp_pre, _ = set_piece_prior(TAGDF, bf["bspeed"], cfg)
    return {**STATIC, **bf, **ps, "sp_pre": sp_pre}

def team_evidence(F, cfg, use_tags=True, use_shape=True):
    dH, dA = np.nan_to_num(F["dH"], nan=60.0), np.nan_to_num(F["dA"], nan=60.0)
    gate = np.exp(-(np.nan_to_num(F["dmin"], nan=60.0) / cfg["r_gate"]) ** 2)
    L = np.clip(cfg["kappa"] * (dA - dH), -cfg["clip"], cfg["clip"]) * gate
    L = np.where(F["vis"], L, 0.0)
    if use_tags: L = L + F["L_TAG"]
    if use_shape: L = L + np.where(F["vis"], 0.0, np.clip(cfg["w_shape"] * logit(F["P_HOME_SHAPE"]), -cfg["shape_clip"], cfg["shape_clip"]))
    return L

def dead_evidence(F, cfg, use_tags=True, use_pm=True):
    LD = np.where(F["dead_ps"], cfg["kin_dead"], 0.0) - np.where(F["alive_ps"], cfg["kin_alive"], 0.0)
    if use_pm: LD = LD + cfg["w_pm"] * np.clip(logit(F["P_DEAD_PM"]), -cfg["pm_clip"], cfg["pm_clip"])
    if use_tags: LD = LD + cfg["w_sp"] * F["sp_pre"]
    return LD

def decode(F, cfg=CFG, use_tags=True, use_shape=True, use_pm=True, lo=0, hi=None):
    hi = N if hi is None else hi
    L = np.nan_to_num(team_evidence(F, cfg, use_tags, use_shape)); LD = np.nan_to_num(dead_evidence(F, cfg, use_tags, use_pm))
    codes = np.full(N, 3, np.int8)
    for a, b in RUNS:
        a2, b2 = max(a, lo), min(b, hi)
        if b2 <= a2: continue
        ev = LD[a2:b2] / FPS
        dead = viterbi2(-0.5 * ev, 0.5 * ev, cfg["lam_dead"]) == 1
        dead = enforce_min_runs(dead, int(cfg["min_dead_s"] * FPS), int(cfg["min_alive_s"] * FPS))
        # rules published in DATA.md / FAQ.md
        f0, f1 = max(HALF2_START, a2), min(KICK2, b2)
        if f1 > f0: dead[f0 - a2:f1 - a2] = True                      # graded-but-dead run before the 2nd-half kickoff
        if a2 <= KICK1 < b2: dead[KICK1 - a2:min(b2, KICK1 + FPS) - a2] = False  # half 1 is graded from the kickoff itself
        out = np.where(dead, 2, 0).astype(np.int8)
        s, e = run_bounds(~dead)
        for ss, ee in zip(s, e):
            evt = L[a2 + ss:a2 + ee] / FPS
            out[ss:ee] = viterbi2(0.5 * evt, -0.5 * evt, cfg["lam_team"])   # state 0 -> H(0), 1 -> A(1)
        codes[a2:b2] = out
    return codes

t0 = time.time()
F_FULL = make_F()
VARIANTS = {
    f"{TEAM_NAME}_full":          dict(use_tags=True,  use_shape=True,  use_pm=True),
    f"{TEAM_NAME}_no_tags":       dict(use_tags=False, use_shape=True,  use_pm=True),
    f"{TEAM_NAME}_ball_only":     dict(use_tags=False, use_shape=False, use_pm=False),
}
PREDS = {name: decode(F_FULL, CFG, **kw) for name, kw in VARIANTS.items()}
print(f"decoded {len(PREDS)} variants in {time.time()-t0:.0f}s")
for n, c in PREDS.items():
    sp = pm.possession_spells(c, RUNS); L_ = np.array([b - a for a, b, _ in sp]) / FPS
    print(f"  {n:28s} possessions {len(sp):4d}  mean {L_.mean():5.1f}s  D share of graded {(c[GRADED]==2).mean():.1%}  X on graded {(c[GRADED]==3).sum()}")

### 3.6 Write and validate the submission
`<TEAM_NAME>_full` is the submission candidate. The file is checked by the same validator the organisers use. **Re-validate the exact file you upload.**

In [ ]:
RES_DIR = REPO / "evaluation" / "results"; RES_DIR.mkdir(exist_ok=True)
SUB_PATH = RES_DIR / f"{TEAM_NAME}.csv"
labels = np.array(list("HADX"))[PREDS[f"{TEAM_NAME}_full"]]
pd.DataFrame({"frame": np.arange(N), "label": labels}).to_csv(SUB_PATH, index=False)
reason = pm.validate_submission(pm.read_label_csv(SUB_PATH), N)
assert reason is None, reason
print("written:", SUB_PATH, f"({SUB_PATH.stat().st_size/1e6:.1f} MB)")
print(subprocess.run([sys.executable, str(REPO / "evaluation" / "validate_submission.py"), str(SUB_PATH)],
                     capture_output=True, text=True).stdout)

## 4 · Evaluation - where do we stand?

What can and cannot be measured:

| evidence | covers | independent of our tuning? | how far to trust it |
|---|---|---|---|
| **A. public ground truth** | 5 min, 17 possessions | yes (we never trained or tuned on it) | *pipeline correctness only* - widest bootstrap interval ≈ 0.3 spellF1 |
| **B. stress test** | the same 5 min, with the ball masked to match the match-wide visibility | yes | best available proxy for the **full-match difficulty** |
| **C. full-match diagnostics** | all graded frames, no labels | n/a | plausibility (counts, durations, fragmentation) |
| **D. Smart-Tagging agreement** | all graded frames, automatic labels | only for the `no_tags` variants | weak independent label; the tags are noisy |
| full-match ground truth | everything | – | **private. Cannot be computed here.** |

### 4A · Public ground truth (official metric code), with bootstrap intervals

In [ ]:
ALL = {**{k: v for k, v in PREDS.items()}, **{k: v for k, v in BASELINES.items()}}
tabA = score_table(ALL, gt, BOOTSTRAP_N)
display(tabA.round(3))
best_base = tabA[tabA.submission.str.startswith("baseline")].iloc[0]
ours = tabA[tabA.submission == f"{TEAM_NAME}_full"].iloc[0]
print(f"ours_full spellF1 {ours.spellF1:.3f} [{ours.spell_lo:.3f}, {ours.spell_hi:.3f}]  vs best baseline on this window "
      f"({best_base.submission}) {best_base.spellF1:.3f} [{best_base.spell_lo:.3f}, {best_base.spell_hi:.3f}]")
overlap = ours.spell_lo <= best_base.spell_hi and best_base.spell_lo <= ours.spell_hi
print("The bootstrap intervals OVERLAP -> this window cannot separate the two." if overlap else
      "The bootstrap intervals do NOT overlap -> a real gap on this window, but still only 17 possessions.")
print("Remember: smooth125 'wins' on this window and loses on the full match (EVALUATION.md). Use this table to check correctness, not to choose methods.")

In [ ]:
rep = possession_report(gt, PREDS[f"{TEAM_NAME}_full"], thr=0.5)
display(rep.round(2))
print(f"matched {rep.matched.sum()} of {len(rep)} true possessions at IoU>=0.5; "
      f"{int((rep.dur_s<4).sum())} true possessions are <4 s and matched {int(rep[rep.dur_s<4].matched.sum())} of them")

# breakdown by ball visibility inside the window: the ball-less frames are where the self-supervised models matter
res = PREDS[f"{TEAM_NAME}_full"][w]; vis_w = VIS[w]
rows = []
for nm, m in (("ball visible", vis_w), ("ball missing", ~vis_w)):
    al = m & (G != pm.D); dd = m & (G == pm.D)
    rows.append(dict(subset=nm, frames=int(m.sum()), team_acc_on_GT_alive=float(((res[al]==G[al])).mean()) if al.any() else np.nan,
                     D_recall=float((res[dd]==pm.D).mean()) if dd.any() else np.nan,
                     D_precision=float((G[m & (res==pm.D)]==pm.D).mean()) if (m & (res==pm.D)).any() else np.nan))
display(pd.DataFrame(rows).round(3))

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    cm_ = ListedColormap(["#d62728", "#1f77b4", "#7f7f7f", "#ffffff"])
    rows = [("ground truth", gt), (f"{TEAM_NAME}_full", PREDS[f"{TEAM_NAME}_full"]), (f"{TEAM_NAME}_ball_only", PREDS[f"{TEAM_NAME}_ball_only"]),
            ("baseline_smooth125", BASELINES["baseline_smooth125"]), ("baseline_smooth50", BASELINES["baseline_smooth50"])]
    fig, ax = plt.subplots(len(rows) + 1, 1, figsize=(13, 0.55 * len(rows) + 1.2), sharex=True)
    for a_, (n_, c_) in zip(ax, rows):
        a_.imshow(c_[WIN_LO:WIN_HI][None, :], aspect="auto", cmap=cm_, vmin=0, vmax=3, extent=[0, (WIN_HI-WIN_LO)/FPS, 0, 1], interpolation="nearest")
        a_.set_yticks([]); a_.set_ylabel(n_, rotation=0, ha="right", va="center", fontsize=8)
    ax[-1].imshow((~VIS[WIN_LO:WIN_HI])[None, :], aspect="auto", cmap="Greys", extent=[0, (WIN_HI-WIN_LO)/FPS, 0, 1], interpolation="nearest")
    ax[-1].set_yticks([]); ax[-1].set_ylabel("ball missing", rotation=0, ha="right", va="center", fontsize=8); ax[-1].set_xlabel("seconds in public window")
    fig.suptitle("red = H, blue = A, grey = D", fontsize=9); plt.tight_layout(); plt.show()
except Exception as e:
    print("plot skipped:", e)

### 4B · Stress test: the public window is unrepresentative, so degrade it
The window has the ball on 88% of frames; the full match has 59%, in gaps of ~17 s. We take the public 5 minutes and **mask the ball** with gaps drawn from the *empirical* gap-length distribution of this match
until the window's visibility matches the match-wide figure, re-run the full pipeline (features → decode), and score against the real labels. Averaged over many random masks and applied identically to the repo's baselines.

*Limit:* real gaps cluster in crowded/aerial phases; random placement cannot reproduce that, so treat this as a better-than-nothing proxy, not a forecast.

In [ ]:
gap_pool = gaps[gaps > 0]                                      # empirical ball-missing gap lengths (frames), graded frames, this match
vis_target = VIS[GRADED].mean(); vis_now = VIS[WIN_LO:WIN_HI].mean()
extra_frac = max(0.0, 1 - vis_target / vis_now)                # fraction of currently-visible window frames to hide
print(f"window visibility {vis_now:.1%} -> target {vis_target:.1%}: hide ~{extra_frac:.0%} of visible window frames, gap lengths drawn from {len(gap_pool)} real gaps")

def make_mask(rs, lo, hi, frac):
    m = np.zeros(N, bool)
    while (m[lo:hi] & VIS[lo:hi]).sum() < frac * VIS[lo:hi].sum():
        g_ = int(rs.choice(gap_pool)); s_ = int(rs.randint(lo, hi))
        m[s_:min(hi, s_ + g_)] = True
    return m

def baseline_masked(BXm, BYm, window):
    rows_ = (max(0, WIN_LO - 200), min(N, WIN_HI + 200))
    c = nearest_team_codes(PX, PY, BXm, BYm, TEAM_OF_SLOT, rows=rows_).astype(np.int8)
    c = np.where(c == 2, -1, c).astype(np.int8)                # repo convention: NOBODY = -1 -> D
    return pm.label_codes(repo_baseline.smooth(c, window))

N_DRAWS = 20
rs = np.random.RandomState(123); acc = {}
t0 = time.time()
for dr in range(N_DRAWS):
    msk = make_mask(rs, WIN_LO, WIN_HI, extra_frac)
    BXm, BYm = BX.copy(), BY.copy(); BXm[msk] = np.nan; BYm[msk] = np.nan
    Fm = make_F(BXm, BYm, rows=(WIN_LO, WIN_HI))
    cands = {"ours_full": decode(Fm, CFG, True, True, True, WIN_LO, WIN_HI),
             "ours_no_tags": decode(Fm, CFG, False, True, True, WIN_LO, WIN_HI),
             "ours_ball_only": decode(Fm, CFG, False, False, False, WIN_LO, WIN_HI),
             "baseline_smooth50": baseline_masked(BXm, BYm, 50), "baseline_smooth125": baseline_masked(BXm, BYm, 125)}
    for n_, c_ in cands.items():
        r_ = score_one(gt, c_)
        acc.setdefault(n_, []).append((r_["spell_f1_mean"], r_["macro_f1"], r_["n_pred_spells"]))
    if dr == 0: print(f"first draw took {time.time()-t0:.0f}s (x{N_DRAWS})")
stress = pd.DataFrame({n_: dict(spellF1_mean=np.mean([x[0] for x in v]), spellF1_sd=np.std([x[0] for x in v]),
                                macroF1_mean=np.mean([x[1] for x in v]), macroF1_sd=np.std([x[1] for x in v]),
                                poss_mean=np.mean([x[2] for x in v])) for n_, v in acc.items()}).T
display(stress.sort_values("spellF1_mean", ascending=False).round(3))
print("(ground truth has 17 possessions in the window). Compare each row's drop vs its unmasked score in 4A.")

### 4C · Full-match diagnostics (no labels needed)
These are plausibility checks on the *whole* graded match. `poss_per_min` is possessions per minute of graded time (including dead ball); the public window gives ~3.4 per minute as a rough
reference (17 possessions in 5 min - a weak prior with a wide interval, in the easiest part of the match), and the baselines show what over-fragmentation looks like (`baseline_raw`: ~1,950 possessions, 80% shorter than 3 s).

In [ ]:
def diag(c):
    sp = pm.possession_spells(c, RUNS); L_ = np.array([b - a for a, b, _ in sp]) / FPS
    gc = c[GRADED]
    return dict(possessions=len(sp), mean_s=L_.mean(), median_s=np.median(L_), p10_s=np.percentile(L_, 10), p90_s=np.percentile(L_, 90),
                share_lt3s=(L_ < 3).mean(), D_share=(gc == 2).mean(), H_share_of_alive=(gc == 0).sum() / max(1, (gc < 2).sum()),
                poss_per_min=len(sp) / (GRADED.sum() / FPS / 60), transitions=len(pm.transitions(c, RUNS)))
dg = pd.DataFrame({n: diag(c) for n, c in {**PREDS, **BASELINES}.items()}).T
display(dg.round(3))

vis_g = VIS[GRADED]; c_full = PREDS[f"{TEAM_NAME}_full"]
by_vis = {}
for nm, m in (("ball visible", VIS), ("ball missing", ~VIS)):
    mm = m & GRADED
    by_vis[nm] = dict(frames=int(mm.sum()), D=(c_full[mm] == 2).mean(), H=(c_full[mm] == 0).mean(), A=(c_full[mm] == 1).mean())
display(pd.DataFrame(by_vis).T.round(3))
print("Our D share on ball-missing frames vs visible frames: a large gap would mean 'ball missing => D' leaked back in (the repo baseline's main flaw).")

### 4D · Agreement with Smart-Tagging team tags across the full match (weak, independent label)

In [ ]:
def tag_agreement(c):
    s = np.sign(L_TAG_RAW); m = GRADED & (s != 0) & (c < 2)
    pred_sign = np.where(c == 0, 1, -1)
    return float((pred_sign[m] == s[m]).mean()), int(m.sum())
rows = []
for n, c in PREDS.items():
    a_, n_ = tag_agreement(c)
    circular = "tags are an input" if n.endswith("_full") else "independent of tags"
    rows.append(dict(variant=n, agreement=a_, frames=n_, note=circular))
display(pd.DataFrame(rows).round(3))
print("Only the *_no_tags / *_ball_only rows are independent evidence. The tags are automatic, so ~100% is neither expected nor desirable.")

### 4E · Sensitivity to the one knob that matters (and why not to tune it here)
`lam_team` (nat·s needed to flip possession) trades fragmentation against missed short possessions. Shown with bootstrap intervals so you can see how little 17 possessions can say.
**Pick it with label-free evidence (4C) and football sense, not by taking the arg-max of this table** - EVALUATION.md's own baseline sweep shows that mistake.

In [ ]:
rows = []
for lam in (1.0, 2.0, 2.5, 4.0, 6.0, 10.0):
    cfg_ = {**CFG, "lam_team": lam}
    c = decode(F_FULL, cfg_)
    r = score_one(gt, c, 300); sp = pm.possession_spells(c, RUNS)
    rows.append(dict(lam_team=lam, full_match_possessions=len(sp), mean_s=np.mean([b - a for a, b, _ in sp]) / FPS,
                     window_spellF1=r["spell_f1_mean"], lo=r["spell_bootstrap_low"], hi=r["spell_bootstrap_high"], window_macroF1=r["macro_f1"]))
display(pd.DataFrame(rows).round(3))

### 4F · The official scorer, run exactly as the organisers will (public ground truth)

In [ ]:
out = subprocess.run([sys.executable, str(REPO / "evaluation" / "run_evaluation.py"), "--bootstrap", "1000"],
                     capture_output=True, text=True, cwd=str(REPO))
print("\n".join(out.stdout.splitlines()[:22]))

### 4G · Reading the results
* **Trust order:** 4F/4A confirm the pipeline is *correct* (labels aligned, file valid, metrics wired). 4B is the closest thing to a full-match forecast. 4C/4D are sanity checks.
  The ranking number - spellF1 on the private full match - is not computable by any participant.
* **Published reference points (full match):** `baseline_smooth50` spellF1 **0.2433** is the floor; `baseline_raw` 0.0486. On the public window the baselines read 0.04–0.42.
* **If 4B shows the ball-only variant collapsing but the full variant holding up,** the ball-less machinery (tags + player-only models) is doing its job; if `ours_no_tags` ≈ `ours_full`, the tags add little and you can drop that dependency.
* **Where to improve next, in expected order of value:** (1) restart handling - dead-ball start/end timing, since each stoppage is a possession boundary; (2) the team-shape model (richer features, e.g. pressure/line height, learned on the whole match with the same pseudo-label trick);
  (3) calibrate `κ`, `r_gate`, `lam_*` with label-free criteria or a held-out *visible-ball* proxy, never by the public score alone; (4) the broadcast ATD as a second ball source (needs frame alignment you must build);
  (5) the video itself (ball detector / homography overlay) for the long ball-less gaps.

### Submit
`evaluation/results/<TEAM_NAME>.csv` - name it after your team, re-run `python evaluation/validate_submission.py` on the exact file, upload with your public repo URL and a short technical document
(approach · assets used/dropped · possession rules · limitations · reproduction). Deadline **17 Nov 2026**. Keep `data/` out of your public repo.